# Injury frequency: negative binomial GLM

**What it predicts:** λ, the expected number of injuries a player has in one big-five first-division season (July to June, 2015/16 onwards).

**Model:** negative binomial (NB2) GLM with a log link, with α fitted by maximum likelihood. Every predictor is known at the start of the season:

```
log E[injuries] = league + position + age + age^2
                  + log(1 + full games last season) + no games last season
                  + log(1 + prior injuries) + log(1 + days out to injuries that started last season)
```

Position is compared against midfielders, league against the Premier League, and age is centred at 26. The counts are overdispersed (the variance is larger than the mean), so a Poisson GLM understates how often players have none or many injuries. That's why the final model is negative binomial: it gives P(no injury) = (1 + αλ)^(−1/α). Injuries are counted the same way as in the severity model. The population is big-five only, because Transfermarkt records injuries densely only there (see [frequency.py](../model/frequency.py)).

In [1]:
import sys; sys.path.insert(0, "..")   # run from runs/: find the model/ and database/ packages
import numpy as np
import pandas as pd
from model import frequency as freq

data = freq.build_dataset()
print(f"{len(data):,} player-seasons, {data.pid.nunique():,} players, {data.n_injuries.sum():,} injuries "
      f"({data.n_injuries.mean():.3f} per season, variance {data.n_injuries.var():.3f})")

30,240 player-seasons, 9,887 players, 31,014 injuries (1.026 per season, variance 1.735)


## Held-out evaluation

Players are split 80/20, and a player held out of the severity model is held out here too. Higher `log_lik` is better, and lower `poisson_dev` and `mae` are better. `mean_pred` and `p_zero` (the predicted share of injury-free seasons) should match the `(actual)` row.

In [2]:
train, test = freq.split(data)
models = freq.fit_all(train)
nb = models[freq.FINAL]
print(f"Poisson GLM Pearson chi2/df = {models['Poisson GLM'].dispersion:.2f}; NB alpha = {nb.alpha:.3f}")
freq.compare(models, test)

  fitting Mean rate, no predictors ...


  fitting Poisson GLM ...


  fitting Negative binomial GLM ...


Poisson GLM Pearson chi2/df = 1.30; NB alpha = 0.218


,log_lik,poisson_dev,mae,mean_pred,p_zero
model,,,,,
"Mean rate, no predictors",-1.3991,1.6530,0.9799,1.0261,0.4675
Poisson GLM,-1.2808,1.2676,0.8538,1.0382,0.4257
Negative binomial GLM,-1.2668,1.2676,0.8536,1.0411,0.4576
(actual),NaN,NaN,NaN,1.0235,0.4722


## Calibration by predicted decile (held-out players)

Player-seasons are sorted by λ into ten equal groups. `pred` vs `actual` compares mean injuries per season, and `p0_pred` vs `p0_actual` compares the share of seasons with no injury.

In [3]:
freq.calibration(nb, test)

,n,pred,actual,p0_pred,p0_actual
decile,,,,,
1,609,0.248,0.186,0.787,0.859
2,609,0.390,0.323,0.688,0.770
3,608,0.486,0.518,0.631,0.664
4,609,0.607,0.695,0.566,0.570
5,608,0.775,0.862,0.490,0.457
6,609,0.965,1.059,0.417,0.394
7,608,1.173,1.146,0.352,0.372
8,609,1.449,1.521,0.285,0.253
9,608,1.790,1.597,0.222,0.243


## Coefficients

Fitted on training players. `rate_ratio` = exp(coef) multiplies the expected number of injuries: 1.30 means 30% more injuries per season, holding the rest fixed. League terms partly reflect how completely each league's injuries are recorded.

In [4]:
freq.coefficients(nb)

,coef,rate_ratio,p_value
variable,,,
Intercept,-1.4036,0.2457,0.0000
league = ES1,-0.0239,0.9764,0.3156
league = IT1,0.2895,1.3358,0.0000
league = L1,0.3094,1.3626,0.0000
league = FR1,-0.1218,0.8853,0.0000
position_group = Goalkeeper,-0.4610,0.6307,0.0000
position_group = Defender,0.0447,1.0457,0.0132
position_group = Attack,0.0878,1.0918,0.0000
age_c,-0.0071,0.9929,0.0014


## More

- [pricing_model.ipynb](pricing_model.ipynb): λ combined with the severity and career-ending models, back-tested at season level.
- Code: [frequency.py](../model/frequency.py).